In [1]:
import os
import json
import time
import joblib
import requests
import numpy as np
import pandas as pd

from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from requests.exceptions import Timeout, ConnectionError, RequestException
from concurrent.futures import ThreadPoolExecutor, as_completed

In [ ]:
API_KEY = "1fd211478dd3b6374a813ec50561eb4e"

MODEL_DIR = "saved_best_models"

OPENWEATHER_AIR_URL = "http://api.openweathermap.org/data/2.5/air_pollution/history"
OPEN_METEO_ARCHIVE_URL = "https://api.open-meteo.com/v1/forecast"

TIMEZONE = "Asia/Makassar"
tz = ZoneInfo(TIMEZONE)

# Ambil data 24 jam ke belakang dari waktu sekarang
end_dt = datetime.now(tz)
start_dt = end_dt - timedelta(hours=24)

end_dt_naive = end_dt.replace(tzinfo=None)
start_dt_naive = start_dt.replace(tzinfo=None)

print("Start datetime:", start_dt_naive.strftime("%Y-%m-%d %H:%M:%S"))
print("End datetime  :", end_dt_naive.strftime("%Y-%m-%d %H:%M:%S"))

# Retry config
max_retries = 5
request_timeout = 30
retry_sleep = 10
sleep_seconds = 1.2

Start datetime: 2026-06-22 23:43:29
End datetime  : 2026-06-23 23:43:29


In [3]:
# =========================
# KOLOM POLUTAN DAN CUACA
# =========================

pollutant_cols = [
    "co", "no", "no2", "o3",
    "so2", "pm2_5", "pm10", "nh3"
]

weather_cols = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "surface_pressure",
    "wind_speed_10m",
    "wind_direction_10m"
]

lag_hours = [1, 3, 6, 12, 24]
rolling_windows = [3, 6, 24]

In [4]:
# =========================
# FUNGSI FETCH DENGAN RETRY
# =========================

def fetch_with_retry(
    url,
    params,
    max_retries=5,
    timeout=30,
    retry_sleep=10,
    last_success_dt=None
):
    for attempt in range(1, max_retries + 1):
        try:
            response = requests.get(url, params=params, timeout=timeout)

            print("      Status code:", response.status_code)

            if response.status_code == 200:
                return response.json()

            print(f"      Status bukan 200. Percobaan {attempt}/{max_retries}")
            print("      Response:", response.text[:500])

        except Timeout:
            print(f"      Timeout. Percobaan {attempt}/{max_retries}")

            if last_success_dt is not None:
                print(f"      Data terakhir berhasil diambil sampai: {last_success_dt}")
            else:
                print("      Belum ada data yang berhasil diambil.")

        except ConnectionError as e:
            print(f"      Koneksi gagal. Percobaan {attempt}/{max_retries}")

            if last_success_dt is not None:
                print(f"      Data terakhir berhasil diambil sampai: {last_success_dt}")
            else:
                print("      Belum ada data yang berhasil diambil.")

            print("      Detail error:", e)

        except RequestException as e:
            print(f"      Request error. Percobaan {attempt}/{max_retries}")

            if last_success_dt is not None:
                print(f"      Data terakhir berhasil diambil sampai: {last_success_dt}")
            else:
                print("      Belum ada data yang berhasil diambil.")

            print("      Detail error:", e)

        if attempt < max_retries:
            print(f"      Menunggu {retry_sleep} detik sebelum retry...")
            time.sleep(retry_sleep)

    return None

In [5]:
# =========================
# BACA FILE JSON KOORDINAT
# =========================

with open("Data/koordinat.json", "r", encoding="utf-8") as f:
    regions = json.load(f)

print("File koordinat.json berhasil dibaca.")

total_points = sum(
    len(points)
    for districts in regions.values()
    for points in districts.values()
)

print("Total kabupaten/kota:", len(regions))
print("Total titik:", total_points)

File koordinat.json berhasil dibaca.
Total kabupaten/kota: 8
Total titik: 588


In [6]:
# =========================
# FLATTEN KOORDINAT JADI LIST TITIK
# =========================

all_points = []

for regency_name, districts in regions.items():
    for district_name, points in districts.items():
        for point_index, point in enumerate(points, start=1):
            all_points.append({
                "regency": regency_name,
                "district": district_name,
                "point_index": point_index,
                "total_points_in_district": len(points),
                "lat": point["lat"],
                "lon": point["lon"],
                "description": point.get("description", ""),
                "point_id": f"{regency_name}_{district_name}_{point_index}"
            })

print("Total titik siap diproses:", len(all_points))

Total titik siap diproses: 588


In [7]:
# =========================
# SCRAPING OPENWEATHER AIR POLLUTION
# 24 JAM TERAKHIR
# =========================

BATCH_SIZE = 10

start_unix = int(start_dt.timestamp())
end_unix = int(end_dt.timestamp())

def fetch_air_point(point):
    lat = point["lat"]
    lon = point["lon"]

    params = {
        "lat": lat,
        "lon": lon,
        "start": start_unix,
        "end": end_unix,
        "appid": API_KEY
    }

    print(
        f"Ambil AQ | {point['regency']} - {point['district']} "
        f"| Titik {point['point_index']} | {lat}, {lon}"
    )

    data = fetch_with_retry(
        url=OPENWEATHER_AIR_URL,
        params=params,
        max_retries=max_retries,
        timeout=request_timeout,
        retry_sleep=retry_sleep,
        last_success_dt=None
    )

    rows = []

    if data is None:
        print(f"  Gagal AQ: {point['point_id']}")
        return rows

    if "list" not in data:
        print(f"  Response AQ tidak memiliki key 'list': {point['point_id']}")
        print("  Response:", data)
        return rows

    if len(data["list"]) == 0:
        print(f"  Data AQ kosong: {point['point_id']}")
        return rows

    for item in data["list"]:
        components = item["components"]

        timestamp = datetime.fromtimestamp(
            item["dt"],
            tz=tz
        ).replace(tzinfo=None)

        rows.append({
            "timestamp": timestamp,
            "regency": point["regency"],
            "district": point["district"],
            "point_id": point["point_id"],
            "description": point["description"],
            "lat": lat,
            "lon": lon,
            "aqi": item["main"]["aqi"],
            "co": components.get("co"),
            "no": components.get("no"),
            "no2": components.get("no2"),
            "o3": components.get("o3"),
            "so2": components.get("so2"),
            "pm2_5": components.get("pm2_5"),
            "pm10": components.get("pm10"),
            "nh3": components.get("nh3")
        })

    if rows:
        print(f"  Berhasil AQ: {point['point_id']} | data: {len(rows)}")

    return rows


air_rows = []

for batch_start in range(0, len(all_points), BATCH_SIZE):
    batch_points = all_points[batch_start:batch_start + BATCH_SIZE]

    print(
        f"\nMemproses batch AQ "
        f"{batch_start + 1} - {batch_start + len(batch_points)} "
        f"dari {len(all_points)} titik"
    )

    with ThreadPoolExecutor(max_workers=BATCH_SIZE) as executor:
        futures = [
            executor.submit(fetch_air_point, point)
            for point in batch_points
        ]

        for future in as_completed(futures):
            try:
                rows = future.result()
                air_rows.extend(rows)
            except Exception as e:
                print("Error saat mengambil data AQ:", e)

    time.sleep(sleep_seconds)

air_df = pd.DataFrame(air_rows)

print("\nJumlah data kualitas udara:", len(air_df))
display(air_df.head())


Memproses batch AQ 1 - 10 dari 588 titik
Ambil AQ | Kota Denpasar - Denpasar Barat | Titik 1 | -8.6516353, 115.1947762
Ambil AQ | Kota Denpasar - Denpasar Barat | Titik 2 | -8.681709, 115.19703
Ambil AQ | Kota Denpasar - Denpasar Barat | Titik 3 | -8.654969, 115.2103865
Ambil AQ | Kota Denpasar - Denpasar Barat | Titik 4 | -8.676031, 115.215224
Ambil AQ | Kota Denpasar - Denpasar Barat | Titik 5 | -8.684573, 115.226236
Ambil AQ | Kota Denpasar - Denpasar Barat | Titik 6 | -8.679993, 115.180557
Ambil AQ | Kota Denpasar - Denpasar Barat | Titik 7 | -8.700204, 115.185217
Ambil AQ | Kota Denpasar - Denpasar Barat | Titik 8 | -8.648762, 115.186305
Ambil AQ | Kota Denpasar - Denpasar Timur | Titik 1 | -8.6483648, 115.2376801
Ambil AQ | Kota Denpasar - Denpasar Timur | Titik 2 | -8.649475, 115.255338
      Status code: 200
  Berhasil AQ: Kota Denpasar_Denpasar Barat_2 | data: 24
      Status code: 200
  Berhasil AQ: Kota Denpasar_Denpasar Barat_3 | data: 24
      Status code: 200
  Berhasil 

,timestamp,regency,district,point_id,description,lat,lon,aqi,co,no,no2,o3,so2,pm2_5,pm10,nh3
0,2026-06-23 00:00:00,Kota Denpasar,Denpasar Barat,Kota Denpasar_Denpasar Barat_2,Perempatan Teuku Umar Barat,-8.681709,115.19703,2,63.96,0.0,0.05,34.51,0.17,5.92,22.60,0.04
1,2026-06-23 01:00:00,Kota Denpasar,Denpasar Barat,Kota Denpasar_Denpasar Barat_2,Perempatan Teuku Umar Barat,-8.681709,115.19703,2,64.20,0.0,0.05,35.12,0.17,5.71,21.65,0.04
2,2026-06-23 02:00:00,Kota Denpasar,Denpasar Barat,Kota Denpasar_Denpasar Barat_2,Perempatan Teuku Umar Barat,-8.681709,115.19703,1,64.77,0.0,0.05,35.87,0.16,5.35,19.81,0.05
3,2026-06-23 03:00:00,Kota Denpasar,Denpasar Barat,Kota Denpasar_Denpasar Barat_2,Perempatan Teuku Umar Barat,-8.681709,115.19703,1,65.76,0.0,0.05,36.68,0.15,5.08,18.07,0.07
4,2026-06-23 04:00:00,Kota Denpasar,Denpasar Barat,Kota Denpasar_Denpasar Barat_2,Perempatan Teuku Umar Barat,-8.681709,115.19703,1,66.94,0.0,0.06,37.59,0.14,4.96,16.48,0.08


In [8]:
# =========================
# SCRAPING OPEN-METEO FORECAST API
# 24 JAM TERAKHIR SAMPAI CURRENT DATETIME
# KOLOM CUACA SESUAI TRAINING
# =========================

BATCH_SIZE = 15

weather_df_list = []

print("Weather start_dt:", start_dt)
print("Weather end_dt  :", end_dt)
print("Weather cols    :", weather_cols)


def fetch_weather_point(point):
    lat = point["lat"]
    lon = point["lon"]

    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": ",".join(weather_cols),
        "timezone": TIMEZONE,
        "past_days": 1,
        "forecast_days": 1
    }

    print(
        f"Ambil Weather | {point['regency']} - {point['district']} "
        f"| Titik {point['point_index']} | {lat}, {lon}"
    )

    data = fetch_with_retry(
        url=OPEN_METEO_ARCHIVE_URL,
        params=params,
        max_retries=max_retries,
        timeout=request_timeout,
        retry_sleep=retry_sleep,
        last_success_dt=None
    )

    if data is None:
        print(f"  Gagal Weather: {point['point_id']}")
        return pd.DataFrame()

    if "hourly" not in data:
        print(f"  Response Weather tidak memiliki key 'hourly': {point['point_id']}")
        print("  Response:", data)
        return pd.DataFrame()

    weather_point_df = pd.DataFrame(data["hourly"])

    if weather_point_df.empty:
        print(f"  Data Weather kosong: {point['point_id']}")
        return pd.DataFrame()

    weather_point_df["timestamp"] = pd.to_datetime(weather_point_df["time"])

    if weather_point_df["timestamp"].dt.tz is None:
        weather_point_df["timestamp"] = weather_point_df["timestamp"].dt.tz_localize(None)

    # Filter hanya 24 jam terakhir sampai current datetime
    weather_point_df = weather_point_df[
        (weather_point_df["timestamp"] >= start_dt_naive) &
        (weather_point_df["timestamp"] <= end_dt_naive)
    ].copy()

    if weather_point_df.empty:
        print(f"  Data Weather kosong setelah filter 24 jam: {point['point_id']}")
        return pd.DataFrame()

    weather_point_df["regency"] = point["regency"]
    weather_point_df["district"] = point["district"]
    weather_point_df["point_id"] = point["point_id"]
    weather_point_df["description"] = point["description"]
    weather_point_df["lat"] = lat
    weather_point_df["lon"] = lon

    print(f"  Berhasil Weather: {point['point_id']} | data: {len(weather_point_df)}")

    return weather_point_df


for batch_start in range(0, len(all_points), BATCH_SIZE):
    batch_points = all_points[batch_start:batch_start + BATCH_SIZE]

    print(
        f"\nMemproses batch Weather "
        f"{batch_start + 1} - {batch_start + len(batch_points)} "
        f"dari {len(all_points)} titik"
    )

    with ThreadPoolExecutor(max_workers=BATCH_SIZE) as executor:
        futures = [
            executor.submit(fetch_weather_point, point)
            for point in batch_points
        ]

        for future in as_completed(futures):
            try:
                weather_point_df = future.result()

                if not weather_point_df.empty:
                    weather_df_list.append(weather_point_df)

            except Exception as e:
                print("Error saat mengambil data Weather:", e)

    time.sleep(sleep_seconds)


if len(weather_df_list) > 0:
    weather_df = pd.concat(weather_df_list, ignore_index=True)
else:
    weather_df = pd.DataFrame()

print("\nJumlah data cuaca:", len(weather_df))
display(weather_df.head())

Weather start_dt: 2026-06-22 23:43:29.398906+08:00
Weather end_dt  : 2026-06-23 23:43:29.398906+08:00
Weather cols    : ['temperature_2m', 'relative_humidity_2m', 'precipitation', 'surface_pressure', 'wind_speed_10m', 'wind_direction_10m']

Memproses batch Weather 1 - 15 dari 588 titik
Ambil Weather | Kota Denpasar - Denpasar Barat | Titik 1 | -8.6516353, 115.1947762
Ambil Weather | Kota Denpasar - Denpasar Barat | Titik 2 | -8.681709, 115.19703
Ambil Weather | Kota Denpasar - Denpasar Barat | Titik 3 | -8.654969, 115.2103865
Ambil Weather | Kota Denpasar - Denpasar Barat | Titik 4 | -8.676031, 115.215224
Ambil Weather | Kota Denpasar - Denpasar Barat | Titik 5 | -8.684573, 115.226236
Ambil Weather | Kota Denpasar - Denpasar Barat | Titik 6 | -8.679993, 115.180557
Ambil Weather | Kota Denpasar - Denpasar Barat | Titik 7 | -8.700204, 115.185217
Ambil Weather | Kota Denpasar - Denpasar Barat | Titik 8 | -8.648762, 115.186305
Ambil Weather | Kota Denpasar - Denpasar Timur | Titik 1 | -8.6

,time,temperature_2m,relative_humidity_2m,precipitation,surface_pressure,wind_speed_10m,wind_direction_10m,timestamp,regency,district,point_id,description,lat,lon
0,2026-06-23T00:00,24.8,90,0.1,1010.3,9.0,85,2026-06-23 00:00:00,Kota Denpasar,Denpasar Barat,Kota Denpasar_Denpasar Barat_6,Jalan Soputan,-8.679993,115.180557
1,2026-06-23T01:00,24.2,94,0.1,1010.3,5.4,86,2026-06-23 01:00:00,Kota Denpasar,Denpasar Barat,Kota Denpasar_Denpasar Barat_6,Jalan Soputan,-8.679993,115.180557
2,2026-06-23T02:00,24.5,91,0.0,1009.6,7.3,80,2026-06-23 02:00:00,Kota Denpasar,Denpasar Barat,Kota Denpasar_Denpasar Barat_6,Jalan Soputan,-8.679993,115.180557
3,2026-06-23T03:00,24.0,94,0.0,1009.4,8.8,55,2026-06-23 03:00:00,Kota Denpasar,Denpasar Barat,Kota Denpasar_Denpasar Barat_6,Jalan Soputan,-8.679993,115.180557
4,2026-06-23T04:00,23.3,93,0.0,1009.5,8.3,46,2026-06-23 04:00:00,Kota Denpasar,Denpasar Barat,Kota Denpasar_Denpasar Barat_6,Jalan Soputan,-8.679993,115.180557


In [9]:
# =========================
# RATA-RATA KUALITAS UDARA
# TITIK → KECAMATAN → KABUPATEN/KOTA
# PER JAM
# =========================

if air_df.empty:
    raise ValueError("air_df kosong. Tidak bisa lanjut ke proses rata-rata dan prediksi.")

air_df["timestamp"] = pd.to_datetime(air_df["timestamp"])
air_df["timestamp_hour"] = air_df["timestamp"].dt.floor("h")

# =========================
# 1. RATA-RATA TITIK → KECAMATAN PER JAM
# =========================

air_district_hourly_avg = (
    air_df
    .groupby(
        ["timestamp_hour", "regency", "district"],
        as_index=False
    )[pollutant_cols]
    .mean()
)

air_district_hourly_avg = air_district_hourly_avg.sort_values(
    by=["timestamp_hour", "regency", "district"]
).reset_index(drop=True)

print("Rata-rata kualitas udara titik → kecamatan per jam selesai.")
print("Total baris kecamatan:", len(air_district_hourly_avg))
display(air_district_hourly_avg.head())


# =========================
# 2. RATA-RATA KECAMATAN → KABUPATEN/KOTA PER JAM
# Metode: equal district average
# Setiap kecamatan punya bobot sama
# =========================

air_regency_hourly_avg = (
    air_district_hourly_avg
    .groupby(
        ["timestamp_hour", "regency"],
        as_index=False
    )[pollutant_cols]
    .mean()
)

air_regency_hourly_avg = air_regency_hourly_avg.sort_values(
    by=["timestamp_hour", "regency"]
).reset_index(drop=True)

print("Rata-rata kualitas udara kecamatan → kabupaten/kota per jam selesai.")
print("Total baris kabupaten/kota:", len(air_regency_hourly_avg))
display(air_regency_hourly_avg.head())

Rata-rata kualitas udara titik → kecamatan per jam selesai.
Total baris kecamatan: 1368


,timestamp_hour,regency,district,co,no,no2,o3,so2,pm2_5,pm10,nh3
0,2026-06-23,Kabupaten Badung,Abiansemal,63.996923,0.0,0.050000,34.301538,0.187692,6.193077,24.581538,0.042308
1,2026-06-23,Kabupaten Badung,Kuta,63.960000,0.0,0.050000,34.510000,0.170000,5.920000,22.600000,0.040000
2,2026-06-23,Kabupaten Badung,Kuta Selatan,63.758462,0.0,0.044615,34.828462,0.173077,5.948462,22.590000,0.031538
3,2026-06-23,Kabupaten Badung,Kuta Utara,63.960000,0.0,0.050000,34.510000,0.170000,5.920000,22.600000,0.040000
4,2026-06-23,Kabupaten Badung,Mengwi,64.220000,0.0,0.050000,34.140000,0.180000,5.970000,23.320000,0.050000


Rata-rata kualitas udara kecamatan → kabupaten/kota per jam selesai.
Total baris kabupaten/kota: 192


,timestamp_hour,regency,co,no,no2,o3,so2,pm2_5,pm10,nh3
0,2026-06-23,Kabupaten Badung,64.008077,0.0,0.049103,34.413077,0.177179,5.998077,23.231667,0.041923
1,2026-06-23,Kabupaten Bangli,65.595192,0.0,0.090962,34.276154,0.192692,6.103846,23.960577,0.087308
2,2026-06-23,Kabupaten Buleleng,73.695214,0.0,0.254957,34.024786,0.189829,4.963077,17.186154,0.301282
3,2026-06-23,Kabupaten Gianyar,65.168108,0.0,0.074941,34.194559,0.183873,5.960843,23.190402,0.075049
4,2026-06-23,Kabupaten Jembrana,68.456923,0.0,0.104000,34.166923,0.152615,5.047231,18.139692,0.150923


In [10]:
# =========================
# RATA-RATA CUACA
# TITIK → KECAMATAN → KABUPATEN/KOTA
# PER JAM
# =========================

if weather_df.empty:
    print("weather_df kosong. Data cuaca tidak akan digunakan.")

    weather_district_hourly_avg = pd.DataFrame(
        columns=["timestamp_hour", "regency", "district"] + weather_cols
    )

    weather_regency_hourly_avg = pd.DataFrame(
        columns=["timestamp_hour", "regency"] + weather_cols
    )

else:
    weather_df["timestamp"] = pd.to_datetime(weather_df["timestamp"])
    weather_df["timestamp_hour"] = weather_df["timestamp"].dt.floor("h")

    # =========================
    # 1. RATA-RATA TITIK → KECAMATAN PER JAM
    # =========================

    weather_district_hourly_avg = (
        weather_df
        .groupby(
            ["timestamp_hour", "regency", "district"],
            as_index=False
        )[weather_cols]
        .mean()
    )

    weather_district_hourly_avg = weather_district_hourly_avg.sort_values(
        by=["timestamp_hour", "regency", "district"]
    ).reset_index(drop=True)

    print("Rata-rata cuaca titik → kecamatan per jam selesai.")
    print("Total baris kecamatan:", len(weather_district_hourly_avg))
    display(weather_district_hourly_avg.head())


    # =========================
    # 2. RATA-RATA KECAMATAN → KABUPATEN/KOTA PER JAM
    # Metode: equal district average
    # Setiap kecamatan punya bobot sama
    # =========================

    weather_regency_hourly_avg = (
        weather_district_hourly_avg
        .groupby(
            ["timestamp_hour", "regency"],
            as_index=False
        )[weather_cols]
        .mean()
    )

    weather_regency_hourly_avg = weather_regency_hourly_avg.sort_values(
        by=["timestamp_hour", "regency"]
    ).reset_index(drop=True)

    print("Rata-rata cuaca kecamatan → kabupaten/kota per jam selesai.")
    print("Total baris kabupaten/kota:", len(weather_regency_hourly_avg))
    display(weather_regency_hourly_avg.head())

Rata-rata cuaca titik → kecamatan per jam selesai.
Total baris kecamatan: 1368


,timestamp_hour,regency,district,temperature_2m,relative_humidity_2m,precipitation,surface_pressure,wind_speed_10m,wind_direction_10m
0,2026-06-23,Kabupaten Badung,Abiansemal,22.923077,98.307692,0.007692,998.392308,4.238462,281.153846
1,2026-06-23,Kabupaten Badung,Kuta,25.046154,90.000000,0.100000,1011.907692,9.000000,85.000000
2,2026-06-23,Kabupaten Badung,Kuta Selatan,26.276923,79.000000,0.100000,1006.061538,17.461538,99.384615
3,2026-06-23,Kabupaten Badung,Kuta Utara,24.430769,93.153846,0.030769,1010.553846,9.207692,78.923077
4,2026-06-23,Kabupaten Badung,Mengwi,22.915385,97.461538,0.000000,998.230769,5.561538,34.307692


Rata-rata cuaca kecamatan → kabupaten/kota per jam selesai.
Total baris kabupaten/kota: 192


,timestamp_hour,regency,temperature_2m,relative_humidity_2m,precipitation,surface_pressure,wind_speed_10m,wind_direction_10m
0,2026-06-23,Kabupaten Badung,23.941026,91.653846,0.039744,1000.033333,8.506410,98.935897
1,2026-06-23,Kabupaten Bangli,19.907692,94.711538,0.001923,934.205769,4.988462,81.269231
2,2026-06-23,Kabupaten Buleleng,25.378632,84.521368,0.000000,1001.104274,2.582906,178.085470
3,2026-06-23,Kabupaten Gianyar,22.198067,93.835994,0.003676,978.660812,6.098256,48.632563
4,2026-06-23,Kabupaten Jembrana,24.835385,87.830769,0.000000,1005.160000,8.269231,101.076923


In [11]:
# =========================
# CEK DAN LENGKAPI MISSING TIMESTAMP
# =========================

def check_missing_timestamp_per_regency(
    df,
    timestamp_col="timestamp_hour",
    freq="1h"
):
    df = df.copy()
    df[timestamp_col] = pd.to_datetime(df[timestamp_col])

    missing_report = []

    for regency, group in df.groupby("regency"):
        group = group.sort_values(timestamp_col)

        existing_times = pd.DatetimeIndex(
            group[timestamp_col].drop_duplicates()
        )

        if existing_times.empty:
            continue

        full_range = pd.date_range(
            start=existing_times.min(),
            end=existing_times.max(),
            freq=freq
        )

        missing_times = full_range.difference(existing_times)

        if len(missing_times) > 0:
            missing_report.append({
                "regency": regency,
                "start_time": existing_times.min(),
                "end_time": existing_times.max(),
                "expected_rows": len(full_range),
                "actual_rows": len(existing_times),
                "missing_count": len(missing_times),
                "missing_timestamps": list(missing_times)
            })

    return pd.DataFrame(missing_report)


def complete_timestamp_per_regency(
    df,
    timestamp_col="timestamp_hour",
    freq="1h"
):
    df = df.copy()
    df[timestamp_col] = pd.to_datetime(df[timestamp_col])

    completed_list = []

    for regency, group in df.groupby("regency"):
        group = group.sort_values(timestamp_col)

        # Hapus duplikasi timestamp-regency agar reindex tidak error
        group = group.drop_duplicates(
            subset=[timestamp_col, "regency"]
        )

        full_range = pd.date_range(
            start=group[timestamp_col].min(),
            end=group[timestamp_col].max(),
            freq=freq
        )

        group = group.set_index(timestamp_col)
        group = group.reindex(full_range)

        group.index.name = timestamp_col

        # Isi kembali nama regency untuk baris timestamp baru
        group["regency"] = regency

        completed_list.append(group.reset_index())

    if len(completed_list) == 0:
        return df

    return pd.concat(completed_list, ignore_index=True)

In [12]:
# =========================
# ISI MISSING VALUE
# METODE: MEDIAN REGENCY + MONTH + HOUR
# =========================

def fill_missing_value_by_regency_month_hour(
    df,
    value_cols,
    timestamp_col="timestamp_hour",
    group_col="regency"
):
    df = df.copy()
    df[timestamp_col] = pd.to_datetime(df[timestamp_col])

    # Fitur bantu untuk median
    df["month"] = df[timestamp_col].dt.month
    df["hour"] = df[timestamp_col].dt.hour

    existing_value_cols = [
        col for col in value_cols
        if col in df.columns
    ]

    print("Kolom numerik yang akan diisi median:")
    print(existing_value_cols)

    # 1. Isi berdasarkan median regency + month + hour
    for col in existing_value_cols:
        df[col] = df[col].fillna(
            df
            .groupby([group_col, "month", "hour"])[col]
            .transform("median")
        )

    # 2. Fallback: median regency + hour
    for col in existing_value_cols:
        df[col] = df[col].fillna(
            df
            .groupby([group_col, "hour"])[col]
            .transform("median")
        )

    # 3. Fallback: median regency
    for col in existing_value_cols:
        df[col] = df[col].fillna(
            df
            .groupby(group_col)[col]
            .transform("median")
        )

    # 4. Fallback: ffill dan bfill per regency
    df = df.sort_values([group_col, timestamp_col]).reset_index(drop=True)

    df[existing_value_cols] = (
        df
        .groupby(group_col)[existing_value_cols]
        .transform(lambda x: x.ffill().bfill())
    )

    # 5. Fallback terakhir agar model tidak error
    df[existing_value_cols] = df[existing_value_cols].fillna(0)

    # Hapus fitur bantu
    df = df.drop(columns=["month", "hour"])

    return df

In [13]:
# =========================
# LENGKAPI AIR POLLUTION
# =========================

if air_regency_hourly_avg.empty:
    raise ValueError("air_regency_hourly_avg kosong. Tidak bisa cek missing timestamp.")

air_missing_report = check_missing_timestamp_per_regency(
    df=air_regency_hourly_avg,
    timestamp_col="timestamp_hour",
    freq="1h"
)

if air_missing_report.empty:
    print("Tidak ada missing timestamp pada air pollution level kabupaten/kota.")
else:
    print("Ditemukan missing timestamp pada air pollution level kabupaten/kota.")
    print("Jumlah kabupaten/kota bermasalah:", len(air_missing_report))

    display(
        air_missing_report[
            [
                "regency",
                "start_time",
                "end_time",
                "expected_rows",
                "actual_rows",
                "missing_count"
            ]
        ]
    )

print("Ukuran air sebelum dilengkapi:", air_regency_hourly_avg.shape)

# Tetap jalankan complete_timestamp_per_regency
# agar struktur timestamp lengkap walaupun report kosong
air_regency_hourly_completed = complete_timestamp_per_regency(
    df=air_regency_hourly_avg,
    timestamp_col="timestamp_hour",
    freq="1h"
)

print("Ukuran air setelah timestamp dilengkapi:", air_regency_hourly_completed.shape)

# Isi nilai polutan yang kosong menggunakan median regency + month + hour
air_regency_hourly_completed = fill_missing_value_by_regency_month_hour(
    df=air_regency_hourly_completed,
    value_cols=pollutant_cols,
    timestamp_col="timestamp_hour",
    group_col="regency"
)

print("Missing value air pollution setelah pengisian:")
display(air_regency_hourly_completed[pollutant_cols].isna().sum())

display(air_regency_hourly_completed.head())

Tidak ada missing timestamp pada air pollution level kabupaten/kota.
Ukuran air sebelum dilengkapi: (192, 10)
Ukuran air setelah timestamp dilengkapi: (192, 10)
Kolom numerik yang akan diisi median:
['co', 'no', 'no2', 'o3', 'so2', 'pm2_5', 'pm10', 'nh3']
Missing value air pollution setelah pengisian:


co       0
no       0
no2      0
o3       0
so2      0
pm2_5    0
pm10     0
nh3      0
dtype: int64

,timestamp_hour,regency,co,no,no2,o3,so2,pm2_5,pm10,nh3
0,2026-06-23 00:00:00,Kabupaten Badung,64.008077,0.0,0.049103,34.413077,0.177179,5.998077,23.231667,0.041923
1,2026-06-23 01:00:00,Kabupaten Badung,64.284744,0.0,0.049103,34.980897,0.173590,5.799872,22.163718,0.044103
2,2026-06-23 02:00:00,Kabupaten Badung,64.885385,0.0,0.049103,35.748718,0.163590,5.460385,20.187308,0.056282
3,2026-06-23 03:00:00,Kabupaten Badung,65.984872,0.0,0.055000,36.636538,0.153590,5.211923,18.336410,0.077051
4,2026-06-23 04:00:00,Kabupaten Badung,67.347051,0.0,0.060769,37.686795,0.145000,5.107564,16.588077,0.097564


In [14]:
# =========================
# LENGKAPI WEATHER
# =========================

if weather_regency_hourly_avg.empty:
    print("weather_regency_hourly_avg kosong. Lewati cek missing timestamp weather.")

    weather_regency_hourly_completed = pd.DataFrame(
        columns=["timestamp_hour", "regency"] + weather_cols
    )

else:
    weather_missing_report = check_missing_timestamp_per_regency(
        df=weather_regency_hourly_avg,
        timestamp_col="timestamp_hour",
        freq="1h"
    )

    if weather_missing_report.empty:
        print("Tidak ada missing timestamp pada weather level kabupaten/kota.")
    else:
        print("Ditemukan missing timestamp pada weather level kabupaten/kota.")
        print("Jumlah kabupaten/kota bermasalah:", len(weather_missing_report))

        display(
            weather_missing_report[
                [
                    "regency",
                    "start_time",
                    "end_time",
                    "expected_rows",
                    "actual_rows",
                    "missing_count"
                ]
            ]
        )

    print("Ukuran weather sebelum dilengkapi:", weather_regency_hourly_avg.shape)

    # Tetap jalankan complete_timestamp_per_regency
    # agar struktur timestamp lengkap walaupun report kosong
    weather_regency_hourly_completed = complete_timestamp_per_regency(
        df=weather_regency_hourly_avg,
        timestamp_col="timestamp_hour",
        freq="1h"
    )

    print("Ukuran weather setelah timestamp dilengkapi:", weather_regency_hourly_completed.shape)

    # Isi nilai weather kosong menggunakan median regency + month + hour
    weather_regency_hourly_completed = fill_missing_value_by_regency_month_hour(
        df=weather_regency_hourly_completed,
        value_cols=weather_cols,
        timestamp_col="timestamp_hour",
        group_col="regency"
    )

    print("Missing value weather setelah pengisian:")
    display(weather_regency_hourly_completed[weather_cols].isna().sum())

    display(weather_regency_hourly_completed.head())

Tidak ada missing timestamp pada weather level kabupaten/kota.
Ukuran weather sebelum dilengkapi: (192, 8)
Ukuran weather setelah timestamp dilengkapi: (192, 8)
Kolom numerik yang akan diisi median:
['temperature_2m', 'relative_humidity_2m', 'precipitation', 'surface_pressure', 'wind_speed_10m', 'wind_direction_10m']
Missing value weather setelah pengisian:


temperature_2m          0
relative_humidity_2m    0
precipitation           0
surface_pressure        0
wind_speed_10m          0
wind_direction_10m      0
dtype: int64

,timestamp_hour,regency,temperature_2m,relative_humidity_2m,precipitation,surface_pressure,wind_speed_10m,wind_direction_10m
0,2026-06-23 00:00:00,Kabupaten Badung,23.941026,91.653846,0.039744,1000.033333,8.506410,98.935897
1,2026-06-23 01:00:00,Kabupaten Badung,23.411538,93.179487,0.038462,999.969231,6.908974,50.307692
2,2026-06-23 02:00:00,Kabupaten Badung,23.612821,91.230769,0.014103,999.366667,7.433333,67.141026
3,2026-06-23 03:00:00,Kabupaten Badung,23.251282,92.038462,0.000000,999.123077,8.294872,68.846154
4,2026-06-23 04:00:00,Kabupaten Badung,22.901282,92.269231,0.000000,999.205128,7.030769,47.602564


In [15]:
# =========================
# GABUNGKAN KUALITAS UDARA + CUACA
# LEVEL KABUPATEN/KOTA PER JAM
# =========================
regency_hourly_input = air_regency_hourly_completed.merge(
    weather_regency_hourly_completed,
    on=["timestamp_hour", "regency"],
    how="left"
)

regency_hourly_input = regency_hourly_input.sort_values(
    by=["regency", "timestamp_hour"]
).reset_index(drop=True)

print("Merge data air pollution dan weather level regency selesai.")
print("Total baris:", len(regency_hourly_input))
print("Jumlah kabupaten/kota:", regency_hourly_input["regency"].nunique())

display(regency_hourly_input.head())

Merge data air pollution dan weather level regency selesai.
Total baris: 192
Jumlah kabupaten/kota: 8


,timestamp_hour,regency,co,no,no2,o3,so2,pm2_5,pm10,nh3,temperature_2m,relative_humidity_2m,precipitation,surface_pressure,wind_speed_10m,wind_direction_10m
0,2026-06-23 00:00:00,Kabupaten Badung,64.008077,0.0,0.049103,34.413077,0.177179,5.998077,23.231667,0.041923,23.941026,91.653846,0.039744,1000.033333,8.506410,98.935897
1,2026-06-23 01:00:00,Kabupaten Badung,64.284744,0.0,0.049103,34.980897,0.173590,5.799872,22.163718,0.044103,23.411538,93.179487,0.038462,999.969231,6.908974,50.307692
2,2026-06-23 02:00:00,Kabupaten Badung,64.885385,0.0,0.049103,35.748718,0.163590,5.460385,20.187308,0.056282,23.612821,91.230769,0.014103,999.366667,7.433333,67.141026
3,2026-06-23 03:00:00,Kabupaten Badung,65.984872,0.0,0.055000,36.636538,0.153590,5.211923,18.336410,0.077051,23.251282,92.038462,0.000000,999.123077,8.294872,68.846154
4,2026-06-23 04:00:00,Kabupaten Badung,67.347051,0.0,0.060769,37.686795,0.145000,5.107564,16.588077,0.097564,22.901282,92.269231,0.000000,999.205128,7.030769,47.602564


In [16]:
# =========================
# FEATURE ENGINEERING LEVEL REGENCY
# TIME, MUSIM, LAG, ROLLING
# =========================

final_input_data = regency_hourly_input.copy()

final_input_data["timestamp_hour"] = pd.to_datetime(
    final_input_data["timestamp_hour"]
)

final_input_data = final_input_data.sort_values(
    by=["regency", "timestamp_hour"]
).reset_index(drop=True)


# =========================
# TIME FEATURES
# =========================

final_input_data["hour"] = final_input_data["timestamp_hour"].dt.hour
final_input_data["day"] = final_input_data["timestamp_hour"].dt.day
final_input_data["month"] = final_input_data["timestamp_hour"].dt.month
final_input_data["dayofweek"] = final_input_data["timestamp_hour"].dt.dayofweek
final_input_data["is_weekend"] = (
    final_input_data["dayofweek"].isin([5, 6]).astype(int)
)


# =========================
# SEASON FEATURE
# Bali / Indonesia:
# April - Oktober  = kemarau = 0
# November - Maret = hujan   = 1
# =========================

def get_season_binary(month):
    if month in [4, 5, 6, 7, 8, 9, 10]:
        return 0
    else:
        return 1

final_input_data["season"] = final_input_data["month"].apply(
    get_season_binary
)


# =========================
# LAG FEATURES
# Level regency: groupby regency
# =========================

for col in pollutant_cols:
    for lag in lag_hours:
        final_input_data[f"{col}_lag_{lag}"] = (
            final_input_data
            .groupby("regency")[col]
            .shift(lag)
        )


# =========================
# ROLLING AVERAGE FEATURES
# Level regency: groupby regency
# shift(1) agar tidak memakai nilai jam yang sama
# =========================

for col in pollutant_cols:
    for window in rolling_windows:
        final_input_data[f"{col}_rolling_avg_{window}"] = (
            final_input_data
            .groupby("regency")[col]
            .transform(
                lambda x: x.shift(1).rolling(
                    window=window,
                    min_periods=1
                ).mean()
            )
        )


print("Feature engineering level regency selesai.")
print("Total kolom:", len(final_input_data.columns))
print("Total baris:", len(final_input_data))

display(final_input_data.head())

Feature engineering level regency selesai.
Total kolom: 86
Total baris: 192


,timestamp_hour,regency,co,no,no2,o3,so2,pm2_5,pm10,nh3,...,so2_rolling_avg_24,pm2_5_rolling_avg_3,pm2_5_rolling_avg_6,pm2_5_rolling_avg_24,pm10_rolling_avg_3,pm10_rolling_avg_6,pm10_rolling_avg_24,nh3_rolling_avg_3,nh3_rolling_avg_6,nh3_rolling_avg_24
0,2026-06-23 00:00:00,Kabupaten Badung,64.008077,0.0,0.049103,34.413077,0.177179,5.998077,23.231667,0.041923,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-06-23 01:00:00,Kabupaten Badung,64.284744,0.0,0.049103,34.980897,0.173590,5.799872,22.163718,0.044103,...,0.177179,5.998077,5.998077,5.998077,23.231667,23.231667,23.231667,0.041923,0.041923,0.041923
2,2026-06-23 02:00:00,Kabupaten Badung,64.885385,0.0,0.049103,35.748718,0.163590,5.460385,20.187308,0.056282,...,0.175385,5.898974,5.898974,5.898974,22.697692,22.697692,22.697692,0.043013,0.043013,0.043013
3,2026-06-23 03:00:00,Kabupaten Badung,65.984872,0.0,0.055000,36.636538,0.153590,5.211923,18.336410,0.077051,...,0.171453,5.752778,5.752778,5.752778,21.860897,21.860897,21.860897,0.047436,0.047436,0.047436
4,2026-06-23 04:00:00,Kabupaten Badung,67.347051,0.0,0.060769,37.686795,0.145000,5.107564,16.588077,0.097564,...,0.166987,5.490726,5.617564,5.617564,20.229145,20.979776,20.979776,0.059145,0.054840,0.054840


In [17]:
# =========================
# ONE-HOT ENCODING REGENCY
# SESUAI TRAINING
# =========================

if "regency" in final_input_data.columns:
    final_input_data["regency_original"] = final_input_data["regency"]

elif "regency_original" in final_input_data.columns:
    print("Kolom regency sudah tidak ada. Menggunakan regency_original.")

else:
    raise ValueError(
        "Kolom 'regency' dan 'regency_original' tidak ditemukan. "
        "Jalankan ulang dari cell merge atau feature engineering."
    )


old_regency_dummy_cols = [
    col for col in final_input_data.columns
    if col.startswith("regency_") and col != "regency_original"
]

if old_regency_dummy_cols:
    final_input_data = final_input_data.drop(columns=old_regency_dummy_cols)


regency_dummies = pd.get_dummies(
    final_input_data["regency_original"],
    prefix="regency",
    drop_first=False
).astype(int)

final_input_data = pd.concat(
    [final_input_data, regency_dummies],
    axis=1
)


regency_dummy_cols = [
    col for col in final_input_data.columns
    if col.startswith("regency_") and col != "regency_original"
]

print("One-hot encoding regency selesai.")
print("Kolom dummy regency:")
print(regency_dummy_cols)

display(final_input_data.head())

One-hot encoding regency selesai.
Kolom dummy regency:
['regency_Kabupaten Badung', 'regency_Kabupaten Bangli', 'regency_Kabupaten Buleleng', 'regency_Kabupaten Gianyar', 'regency_Kabupaten Jembrana', 'regency_Kabupaten Karangasem', 'regency_Kabupaten Klungkung', 'regency_Kota Denpasar']


,timestamp_hour,regency,co,no,no2,o3,so2,pm2_5,pm10,nh3,...,nh3_rolling_avg_24,regency_original,regency_Kabupaten Badung,regency_Kabupaten Bangli,regency_Kabupaten Buleleng,regency_Kabupaten Gianyar,regency_Kabupaten Jembrana,regency_Kabupaten Karangasem,regency_Kabupaten Klungkung,regency_Kota Denpasar
0,2026-06-23 00:00:00,Kabupaten Badung,64.008077,0.0,0.049103,34.413077,0.177179,5.998077,23.231667,0.041923,...,NaN,Kabupaten Badung,1,0,0,0,0,0,0,0
1,2026-06-23 01:00:00,Kabupaten Badung,64.284744,0.0,0.049103,34.980897,0.173590,5.799872,22.163718,0.044103,...,0.041923,Kabupaten Badung,1,0,0,0,0,0,0,0
2,2026-06-23 02:00:00,Kabupaten Badung,64.885385,0.0,0.049103,35.748718,0.163590,5.460385,20.187308,0.056282,...,0.043013,Kabupaten Badung,1,0,0,0,0,0,0,0
3,2026-06-23 03:00:00,Kabupaten Badung,65.984872,0.0,0.055000,36.636538,0.153590,5.211923,18.336410,0.077051,...,0.047436,Kabupaten Badung,1,0,0,0,0,0,0,0
4,2026-06-23 04:00:00,Kabupaten Badung,67.347051,0.0,0.060769,37.686795,0.145000,5.107564,16.588077,0.097564,...,0.054840,Kabupaten Badung,1,0,0,0,0,0,0,0


In [18]:
# =========================
# GUNAKAN SEMUA TIMESTAMP 24 JAM TERAKHIR
# LEVEL REGENCY SEBAGAI INPUT MODEL
# =========================

regency_prediction_input = final_input_data.copy()

print("Total baris input prediksi:", len(regency_prediction_input))
print("Rentang timestamp:")
print(regency_prediction_input["timestamp_hour"].min())
print(regency_prediction_input["timestamp_hour"].max())

print("Jumlah kabupaten/kota:", regency_prediction_input["regency_original"].nunique())

display(regency_prediction_input.head())

Total baris input prediksi: 192
Rentang timestamp:
2026-06-23 00:00:00
2026-06-23 23:00:00
Jumlah kabupaten/kota: 8


,timestamp_hour,regency,co,no,no2,o3,so2,pm2_5,pm10,nh3,...,nh3_rolling_avg_24,regency_original,regency_Kabupaten Badung,regency_Kabupaten Bangli,regency_Kabupaten Buleleng,regency_Kabupaten Gianyar,regency_Kabupaten Jembrana,regency_Kabupaten Karangasem,regency_Kabupaten Klungkung,regency_Kota Denpasar
0,2026-06-23 00:00:00,Kabupaten Badung,64.008077,0.0,0.049103,34.413077,0.177179,5.998077,23.231667,0.041923,...,NaN,Kabupaten Badung,1,0,0,0,0,0,0,0
1,2026-06-23 01:00:00,Kabupaten Badung,64.284744,0.0,0.049103,34.980897,0.173590,5.799872,22.163718,0.044103,...,0.041923,Kabupaten Badung,1,0,0,0,0,0,0,0
2,2026-06-23 02:00:00,Kabupaten Badung,64.885385,0.0,0.049103,35.748718,0.163590,5.460385,20.187308,0.056282,...,0.043013,Kabupaten Badung,1,0,0,0,0,0,0,0
3,2026-06-23 03:00:00,Kabupaten Badung,65.984872,0.0,0.055000,36.636538,0.153590,5.211923,18.336410,0.077051,...,0.047436,Kabupaten Badung,1,0,0,0,0,0,0,0
4,2026-06-23 04:00:00,Kabupaten Badung,67.347051,0.0,0.060769,37.686795,0.145000,5.107564,16.588077,0.097564,...,0.054840,Kabupaten Badung,1,0,0,0,0,0,0,0


In [19]:
# =========================
# LOAD METADATA MODEL
# =========================

metadata_path = os.path.join(MODEL_DIR, "metadata.json")

with open(metadata_path, "r", encoding="utf-8") as f:
    metadata = json.load(f)

feature_names = metadata["feature_names"]
target_pollutants = metadata["pollutant_cols"]

print("Jumlah fitur model:", len(feature_names))
print("Target polutan:", target_pollutants)

print("\nFitur model:")
print(feature_names)

Jumlah fitur model: 84
Target polutan: ['co', 'no', 'no2', 'o3', 'so2', 'pm2_5', 'pm10', 'nh3']

Fitur model:
['temperature_2m', 'relative_humidity_2m', 'precipitation', 'surface_pressure', 'wind_speed_10m', 'wind_direction_10m', 'hour', 'dayofweek', 'day', 'month', 'is_weekend', 'season', 'co_lag_1', 'co_lag_3', 'co_lag_6', 'co_lag_12', 'co_lag_24', 'no_lag_1', 'no_lag_3', 'no_lag_6', 'no_lag_12', 'no_lag_24', 'no2_lag_1', 'no2_lag_3', 'no2_lag_6', 'no2_lag_12', 'no2_lag_24', 'o3_lag_1', 'o3_lag_3', 'o3_lag_6', 'o3_lag_12', 'o3_lag_24', 'so2_lag_1', 'so2_lag_3', 'so2_lag_6', 'so2_lag_12', 'so2_lag_24', 'pm2_5_lag_1', 'pm2_5_lag_3', 'pm2_5_lag_6', 'pm2_5_lag_12', 'pm2_5_lag_24', 'pm10_lag_1', 'pm10_lag_3', 'pm10_lag_6', 'pm10_lag_12', 'pm10_lag_24', 'nh3_lag_1', 'nh3_lag_3', 'nh3_lag_6', 'nh3_lag_12', 'nh3_lag_24', 'co_rolling_avg_3', 'co_rolling_avg_6', 'co_rolling_avg_24', 'no_rolling_avg_3', 'no_rolling_avg_6', 'no_rolling_avg_24', 'no2_rolling_avg_3', 'no2_rolling_avg_6', 'no2_roll

In [20]:
# =========================
# SIAPKAN INPUT SESUAI FITUR TRAINING
# LEVEL REGENCY
# =========================

model_input_df = regency_prediction_input.copy()

missing_features = [
    col for col in feature_names
    if col not in model_input_df.columns
]

if missing_features:
    print("PERINGATAN: fitur berikut belum tersedia dan akan dibuat:")
    print(missing_features)

    for col in missing_features:

        if col == "is_weekend":
            model_input_df[col] = (
                pd.to_datetime(model_input_df["timestamp_hour"])
                .dt.dayofweek
                .isin([5, 6])
                .astype(int)
            )

        elif col.startswith("regency_"):
            regency_name = col.replace("regency_", "", 1)

            model_input_df[col] = (
                model_input_df["regency_original"] == regency_name
            ).astype(int)

        else:
            print(f"Fitur {col} tidak dikenali. Diisi 0 sebagai fallback.")
            model_input_df[col] = 0


# Pastikan semua dummy regency sesuai nama asli regency
for col in feature_names:
    if col.startswith("regency_"):
        regency_name = col.replace("regency_", "", 1)

        model_input_df[col] = (
            model_input_df["regency_original"] == regency_name
        ).astype(int)


numeric_cols = model_input_df.select_dtypes(
    include=[np.number]
).columns

model_input_df[numeric_cols] = (
    model_input_df[numeric_cols]
    .fillna(model_input_df[numeric_cols].mean())
)

model_input_df[numeric_cols] = (
    model_input_df[numeric_cols]
    .fillna(0)
)


X_new = model_input_df[feature_names]

print("Input model level regency siap.")
print("Shape X_new:", X_new.shape)

display(X_new.head())

Input model level regency siap.
Shape X_new: (192, 84)


,temperature_2m,relative_humidity_2m,precipitation,surface_pressure,wind_speed_10m,wind_direction_10m,hour,dayofweek,day,month,...,nh3_rolling_avg_6,nh3_rolling_avg_24,regency_Kabupaten Badung,regency_Kabupaten Bangli,regency_Kabupaten Buleleng,regency_Kabupaten Gianyar,regency_Kabupaten Jembrana,regency_Kabupaten Karangasem,regency_Kabupaten Klungkung,regency_Kota Denpasar
0,23.941026,91.653846,0.039744,1000.033333,8.506410,98.935897,0,1,23,6,...,0.230651,0.198085,1,0,0,0,0,0,0,0
1,23.411538,93.179487,0.038462,999.969231,6.908974,50.307692,1,1,23,6,...,0.041923,0.041923,1,0,0,0,0,0,0,0
2,23.612821,91.230769,0.014103,999.366667,7.433333,67.141026,2,1,23,6,...,0.043013,0.043013,1,0,0,0,0,0,0,0
3,23.251282,92.038462,0.000000,999.123077,8.294872,68.846154,3,1,23,6,...,0.047436,0.047436,1,0,0,0,0,0,0,0
4,22.901282,92.269231,0.000000,999.205128,7.030769,47.602564,4,1,23,6,...,0.054840,0.054840,1,0,0,0,0,0,0,0


In [26]:
# =========================
# PREDIKSI LEVEL REGENCY
# OUTPUT 24 JAM KE DEPAN PER JAM
# =========================

# Pastikan urut berdasarkan regency dan timestamp
regency_prediction_input = regency_prediction_input.sort_values(
    ["regency_original", "timestamp_hour"]
)

# Samakan urutan model_input_df dengan regency_prediction_input
model_input_df = model_input_df.loc[regency_prediction_input.index].copy()

# Ambil 24 timestamp terakhir per regency
latest_24_idx = (
    regency_prediction_input
    .groupby("regency_original", group_keys=False)
    .tail(24)
    .index
)

latest_24_prediction_input = regency_prediction_input.loc[latest_24_idx].copy()
latest_24_model_input_df = model_input_df.loc[latest_24_idx].copy()

# Input model = 24 timestamp terakhir per regency
X_latest_24 = latest_24_model_input_df[feature_names].copy()

# Output timestamp = timestamp input + 24 jam
prediction_result = latest_24_prediction_input[
    [
        "timestamp_hour",
        "regency_original"
    ]
].copy()

prediction_result = prediction_result.rename(
    columns={
        "regency_original": "regency"
    }
)

prediction_result["timestamp_hour"] = (
    prediction_result["timestamp_hour"] + pd.Timedelta(hours=24)
)

for pollutant in target_pollutants:
    model_path = os.path.join(MODEL_DIR, f"{pollutant}_best_model.joblib")

    if not os.path.exists(model_path):
        print(f"Model untuk {pollutant} tidak ditemukan: {model_path}")
        continue

    print(f"Memuat model {pollutant}...")

    model = joblib.load(model_path)

    pred_values = model.predict(X_latest_24)

    # Konsentrasi polutan tidak boleh negatif
    pred_values = np.maximum(pred_values, 0)

    prediction_result[f"{pollutant}_pred_24h"] = pred_values


prediction_cols = [
    "timestamp_hour",
    "regency",
] + [
    f"{pollutant}_pred_24h"
    for pollutant in target_pollutants
    if f"{pollutant}_pred_24h" in prediction_result.columns
]

prediction_result = prediction_result[prediction_cols]

prediction_result = prediction_result.sort_values(
    ["timestamp_hour", "regency"]
).reset_index(drop=True)

display(prediction_result)

Memuat model co...
Memuat model no...
Memuat model no2...
Memuat model o3...
Memuat model so2...
Memuat model pm2_5...
Memuat model pm10...
Memuat model nh3...


,timestamp_hour,regency,co_pred_24h,no_pred_24h,no2_pred_24h,o3_pred_24h,so2_pred_24h,pm2_5_pred_24h,pm10_pred_24h,nh3_pred_24h
0,2026-06-24 00:00:00,Kabupaten Badung,80.699426,0.015169,0.159923,42.697472,0.381884,13.983739,18.435005,0.225660
1,2026-06-24 00:00:00,Kabupaten Bangli,80.594767,0.019988,0.168698,42.926372,0.382714,14.174971,18.317841,0.274324
2,2026-06-24 00:00:00,Kabupaten Buleleng,84.721645,0.029004,0.404853,42.901956,0.390134,14.244692,17.938369,0.381402
3,2026-06-24 00:00:00,Kabupaten Gianyar,80.670139,0.017107,0.162235,42.832432,0.389389,14.042817,18.289647,0.230753
4,2026-06-24 00:00:00,Kabupaten Jembrana,80.823717,0.011872,0.161101,42.778472,0.381482,13.993350,18.452392,0.230000
...,...,...,...,...,...,...,...,...,...,...
187,2026-06-24 23:00:00,Kabupaten Gianyar,81.701544,0.000000,0.115157,42.769760,0.300941,17.724154,20.475604,0.165000
188,2026-06-24 23:00:00,Kabupaten Jembrana,81.493357,0.000000,0.123877,42.908468,0.353560,20.359256,20.560095,0.160455
189,2026-06-24 23:00:00,Kabupaten Karangasem,98.857057,0.000000,0.116973,42.996296,0.320038,17.739033,21.868806,0.202379
190,2026-06-24 23:00:00,Kabupaten Klungkung,86.795169,0.000000,0.090996,43.470582,0.268962,18.398551,21.378582,0.263915


In [27]:
# =========================
# FUNGSI KONVERSI SATUAN DAN BREAKPOINT AQI
# =========================

MW = {
    "co": 28.01,
    "no2": 46.0055,
    "o3": 48.00,
    "so2": 64.066,
}

def ugm3_to_ppb(x, mw):
    return x * 24.45 / mw

def ugm3_to_ppm(x, mw):
    return x * 24.45 / (mw * 1000)


BP_PM25 = [
    (0.0, 12.0, 0, 50),
    (12.1, 35.4, 51, 100),
    (35.5, 55.4, 101, 150),
    (55.5, 150.4, 151, 200),
    (150.5, 250.4, 201, 300),
    (250.5, 350.4, 301, 400),
    (350.5, 500.4, 401, 500),
]

BP_PM10 = [
    (0, 54, 0, 50),
    (55, 154, 51, 100),
    (155, 254, 101, 150),
    (255, 354, 151, 200),
    (355, 424, 201, 300),
    (425, 504, 301, 400),
    (505, 604, 401, 500),
]

BP_CO = [
    (0.0, 4.4, 0, 50),
    (4.5, 9.4, 51, 100),
    (9.5, 12.4, 101, 150),
    (12.5, 15.4, 151, 200),
    (15.5, 30.4, 201, 300),
    (30.5, 40.4, 301, 400),
    (40.5, 50.4, 401, 500),
]

BP_O3_8H = [
    (0.000, 0.054, 0, 50),
    (0.055, 0.070, 51, 100),
    (0.071, 0.085, 101, 150),
    (0.086, 0.105, 151, 200),
    (0.106, 0.200, 201, 300),
]

BP_O3_1H = [
    (0.125, 0.164, 101, 150),
    (0.165, 0.204, 151, 200),
    (0.205, 0.404, 201, 300),
    (0.405, 0.504, 301, 400),
    (0.505, 0.604, 401, 500),
]

BP_SO2 = [
    (0, 35, 0, 50),
    (36, 75, 51, 100),
    (76, 185, 101, 150),
    (186, 304, 151, 200),
    (305, 604, 201, 300),
    (605, 804, 301, 400),
    (805, 1004, 401, 500),
]

BP_NO2 = [
    (0, 53, 0, 50),
    (54, 100, 51, 100),
    (101, 360, 101, 150),
    (361, 649, 151, 200),
    (650, 1249, 201, 300),
    (1250, 1649, 301, 400),
    (1650, 2049, 401, 500),
]


def calc_aqi(C, breakpoints):
    if pd.isna(C):
        return np.nan

    for c_low, c_high, i_low, i_high in breakpoints:
        if c_low <= C <= c_high:
            return round(
                ((i_high - i_low) / (c_high - c_low)) * (C - c_low) + i_low
            )

    return np.nan


def aqi_category(aqi):
    if pd.isna(aqi):
        return np.nan
    elif aqi <= 50:
        return "Good"
    elif aqi <= 100:
        return "Moderate"
    elif aqi <= 150:
        return "Unhealthy for Sensitive Groups"
    elif aqi <= 200:
        return "Unhealthy"
    elif aqi <= 300:
        return "Very Unhealthy"
    else:
        return "Hazardous"

print("Fungsi AQI dan breakpoint berhasil dibuat.")

Fungsi AQI dan breakpoint berhasil dibuat.


In [28]:
# =========================
# HITUNG AQI INDEX DARI HASIL PREDIKSI REGENCY
# =========================

aqi_prediction_df = prediction_result.copy()

for pollutant in target_pollutants:
    pred_col = f"{pollutant}_pred_24h"

    if pred_col in aqi_prediction_df.columns:
        aqi_prediction_df[pollutant] = aqi_prediction_df[pred_col]


required_aqi_pollutants = [
    "co", "no2", "o3", "so2", "pm2_5", "pm10"
]

missing_aqi_pollutants = [
    col for col in required_aqi_pollutants
    if col not in aqi_prediction_df.columns
]

if missing_aqi_pollutants:
    print("AQI tidak bisa dihitung lengkap karena kolom berikut tidak ada:")
    print(missing_aqi_pollutants)

    prediction_result["aqi_score_pred_24h"] = np.nan
    prediction_result["dominant_pollutant_pred_24h"] = np.nan
    prediction_result["aqi_index_pred_24h"] = np.nan

else:
    aqi_prediction_df["co_ppm"] = ugm3_to_ppm(
        aqi_prediction_df["co"],
        MW["co"]
    )

    aqi_prediction_df["no2_ppb"] = ugm3_to_ppb(
        aqi_prediction_df["no2"],
        MW["no2"]
    )

    aqi_prediction_df["o3_ppb"] = ugm3_to_ppb(
        aqi_prediction_df["o3"],
        MW["o3"]
    )

    aqi_prediction_df["so2_ppb"] = ugm3_to_ppb(
        aqi_prediction_df["so2"],
        MW["so2"]
    )

    aqi_prediction_df["o3_ppm"] = aqi_prediction_df["o3_ppb"] / 1000

    aqi_prediction_df["aqi_pm2_5"] = aqi_prediction_df["pm2_5"].apply(
        lambda x: calc_aqi(x, BP_PM25)
    )

    aqi_prediction_df["aqi_pm10"] = aqi_prediction_df["pm10"].apply(
        lambda x: calc_aqi(x, BP_PM10)
    )

    aqi_prediction_df["aqi_co"] = aqi_prediction_df["co_ppm"].apply(
        lambda x: calc_aqi(x, BP_CO)
    )

    aqi_prediction_df["aqi_o3_8h"] = aqi_prediction_df["o3_ppm"].apply(
        lambda x: calc_aqi(x, BP_O3_8H)
    )

    aqi_prediction_df["aqi_o3_1h"] = aqi_prediction_df["o3_ppm"].apply(
        lambda x: calc_aqi(x, BP_O3_1H)
    )

    aqi_prediction_df["aqi_o3"] = aqi_prediction_df[
        ["aqi_o3_8h", "aqi_o3_1h"]
    ].max(axis=1)

    aqi_prediction_df["aqi_so2"] = aqi_prediction_df["so2_ppb"].apply(
        lambda x: calc_aqi(x, BP_SO2)
    )

    aqi_prediction_df["aqi_no2"] = aqi_prediction_df["no2_ppb"].apply(
        lambda x: calc_aqi(x, BP_NO2)
    )

    aqi_cols = [
        "aqi_pm2_5",
        "aqi_pm10",
        "aqi_co",
        "aqi_o3",
        "aqi_so2",
        "aqi_no2"
    ]

    aqi_prediction_df["aqi_score_pred_24h"] = (
        aqi_prediction_df[aqi_cols].max(axis=1)
    )

    aqi_prediction_df["dominant_pollutant_pred_24h"] = (
        aqi_prediction_df[aqi_cols]
        .idxmax(axis=1)
        .str.replace("aqi_", "", regex=False)
    )

    aqi_prediction_df["aqi_index_pred_24h"] = (
        aqi_prediction_df["aqi_score_pred_24h"].apply(aqi_category)
    )

    prediction_result["aqi_score_pred_24h"] = (
        aqi_prediction_df["aqi_score_pred_24h"]
    )

    prediction_result["dominant_pollutant_pred_24h"] = (
        aqi_prediction_df["dominant_pollutant_pred_24h"]
    )

    prediction_result["aqi_index_pred_24h"] = (
        aqi_prediction_df["aqi_index_pred_24h"]
    )


prediction_cols_with_aqi = [
    "timestamp_hour",
    "regency",
] + [
    f"{pollutant}_pred_24h"
    for pollutant in target_pollutants
    if f"{pollutant}_pred_24h" in prediction_result.columns
] + [
    "aqi_score_pred_24h",
    "dominant_pollutant_pred_24h",
    "aqi_index_pred_24h"
]

prediction_result_with_aqi = prediction_result[
    prediction_cols_with_aqi
].copy()

output_filename_aqi = "prediction_24h_aqi.csv"

prediction_result_with_aqi.to_csv(output_filename_aqi, index=False)

print("Perhitungan AQI selesai.")
print("Hasil prediksi kabupaten/kota dengan AQI berhasil disimpan.")
print("Nama file:", output_filename_aqi)
print("Total baris:", len(prediction_result_with_aqi))

display(prediction_result_with_aqi)

Perhitungan AQI selesai.
Hasil prediksi kabupaten/kota dengan AQI berhasil disimpan.
Nama file: prediction_24h_aqi.csv
Total baris: 192


,timestamp_hour,regency,co_pred_24h,no_pred_24h,no2_pred_24h,o3_pred_24h,so2_pred_24h,pm2_5_pred_24h,pm10_pred_24h,nh3_pred_24h,aqi_score_pred_24h,dominant_pollutant_pred_24h,aqi_index_pred_24h
0,2026-06-24 00:00:00,Kabupaten Badung,80.699426,0.015169,0.159923,42.697472,0.381884,13.983739,18.435005,0.225660,55.0,pm2_5,Moderate
1,2026-06-24 00:00:00,Kabupaten Bangli,80.594767,0.019988,0.168698,42.926372,0.382714,14.174971,18.317841,0.274324,55.0,pm2_5,Moderate
2,2026-06-24 00:00:00,Kabupaten Buleleng,84.721645,0.029004,0.404853,42.901956,0.390134,14.244692,17.938369,0.381402,56.0,pm2_5,Moderate
3,2026-06-24 00:00:00,Kabupaten Gianyar,80.670139,0.017107,0.162235,42.832432,0.389389,14.042817,18.289647,0.230753,55.0,pm2_5,Moderate
4,2026-06-24 00:00:00,Kabupaten Jembrana,80.823717,0.011872,0.161101,42.778472,0.381482,13.993350,18.452392,0.230000,55.0,pm2_5,Moderate
...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,2026-06-24 23:00:00,Kabupaten Gianyar,81.701544,0.000000,0.115157,42.769760,0.300941,17.724154,20.475604,0.165000,63.0,pm2_5,Moderate
188,2026-06-24 23:00:00,Kabupaten Jembrana,81.493357,0.000000,0.123877,42.908468,0.353560,20.359256,20.560095,0.160455,68.0,pm2_5,Moderate
189,2026-06-24 23:00:00,Kabupaten Karangasem,98.857057,0.000000,0.116973,42.996296,0.320038,17.739033,21.868806,0.202379,63.0,pm2_5,Moderate
190,2026-06-24 23:00:00,Kabupaten Klungkung,86.795169,0.000000,0.090996,43.470582,0.268962,18.398551,21.378582,0.263915,64.0,pm2_5,Moderate
